## 1.3 BER 扫描与参数分析

在上一节中，我们完成了 GFSK 调制解调的完整链路。本节首先通过 BER-SNR 扫描验证仿真与理论值的一致性，然后探究 GFSK 的两个关键参数——每符号采样点数 **sps** 和调制指数 **mod_index**——在固定 SNR 下对误码率的影响规律。

本节学习大纲如下：

- BER-SNR 扫描与理论验证
- sps / mod_index 参数矩阵扫描

---

### 1. 准备工作

生成随机比特序列和调制、解调器。

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.phy.gfsk import GFSKModulator, GFSKDemodulator
from nearlink_sdr.phy.channel import ChannelModel

sps = 8
num_bits = 10000
rng = np.random.default_rng(42)
tx_bits = rng.integers(0, 2, num_bits)
mod = GFSKModulator(sps=sps)
demod = GFSKDemodulator(sps=sps)

---

### 2. BER-SNR 扫描

遍历多个 SNR 点，统计每个 SNR 下的 GFSK 误码率，观察 BER 随 SNR 的变化趋势。


In [ ]:
snr_range = np.arange(0, 16, 2)
ber_list = []
for snr in snr_range:
    tx_signal = mod.modulate(tx_bits)
    ch = ChannelModel(snr_db=float(snr))
    rx_signal = ch.apply_awgn(tx_signal)
    rx_bits = demod.demodulate(rx_signal)
    n = min(len(tx_bits), len(rx_bits))
    ber = np.mean(tx_bits[:n] != rx_bits[:n])
    ber_list.append(ber)
    print(f"SNR={snr:2d} dB  BER={ber:.6f}")

---

### 3. 与理论值对比

GFSK 本质上是在 BFSK（二进制频移键控）基础上加了高斯预滤波，BFSK 的理论值是 GFSK 的自然参照——对比可验证仿真是否正确，也能看出高斯滤波对性能的影响。

**相干 BFSK** 的理论 BER 公式为：

$$\text{BER} = Q\left(\sqrt{\frac{E_b}{N_0}}\right), \quad Q(x) = \frac{1}{2} \text{erfc}\left(\frac{x}{\sqrt{2}}\right)$$

由于**高斯滤波引入码间干扰**：GFSK 对基带信号进行了高斯低通滤波，平滑了频率过渡但也使相邻符号间产生轻微干扰，等效 SNR 略有损失。**理论值为性能上界**：$Q(\sqrt{E_b/N_0})$ 是相干 BFSK 在理想条件下的 BER 下界，GFSK 作为非理想系统必然高于此值。

将仿真值与理论值逐点对比：

In [ ]:
from scipy.special import erfc

print(f"{'SNR':>5s}  {'SimBER':>10s}  {'TheoryBER':>10s}")
for snr, b in zip(snr_range, ber_list):
    snr_lin = 10**(snr/10)
    t = 0.5 * erfc(np.sqrt(snr_lin))
    print(f"{snr:5.0f}  {b:10.6f}  {t:10.6f}")

---

### 4. 绘制 BER-SNR 曲线

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(snr_range, [max(b, 1e-6) for b in ber_list], "o-", label="GFSK sim")
theory = [0.5 * erfc(np.sqrt(10**(s/10))) for s in snr_range]
ax.semilogy(snr_range, [max(b, 1e-6) for b in theory], "s--", label="Coherent BFSK theory")
ax.set_xlabel("Eb/N0 (dB)")
ax.set_ylabel("Bit Error Rate")
ax.set_title("GFSK BER in AWGN")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.5)
ax.set_ylim(bottom=1e-5)
plt.show()

---

### 5. sps 与 mod_index 对 BER 的影响

GFSK 有两个关键参数直接影响链路性能：

**sps（Samples Per Symbol，每符号采样点数）**：决定每个比特用多少个 IQ 采样点表示。sps 越高，波形越精细，频率分辨率越好，但采样点总数增加，计算量增大，GFSK 通过 `np.repeat` 上采样。

**mod_index（调制指数）**：控制比特 0/1 对应的频率偏移量。标准范围为 [0.45, 0.55]。mod_index 越大，0 和 1 的频率差越大，解调识别越容易，但占用带宽也越宽。

下面在固定 SNR=8 dB 下扫描 sps 和 mod_index 的组合，观察其对 BER 的影响。

In [ ]:
snr_db = 8.0

def measure(sps_val, mi):
    m = GFSKModulator(sps=sps_val, mod_index=mi)
    d = GFSKDemodulator(sps=sps_val)
    tx = m.modulate(tx_bits)
    rx_sig = ChannelModel(snr_db=snr_db).apply_awgn(tx)
    rx_bits = d.demodulate(rx_sig)
    n = min(len(tx_bits), len(rx_bits))
    return np.mean(tx_bits[:n] != rx_bits[:n])

sps_list = [4, 8, 16]
mod_list = [0.45, 0.50, 0.55]

print(f"BER matrix (SNR={snr_db:.0f} dB):")
print(f"{'':>8s}", end="")
for m in mod_list:
    print(f"{'mod=' + str(m):>12s}", end="")
print()
print("-" * 44)
for s in sps_list:
    print(f"{'sps=' + str(s):>8s}", end="")
    for m in mod_list:
        print(f"{measure(s, m):12.6f}", end="")
    print()

得到结果后，不难分析出：mod_index越大，BER 越低。因为调制指数增大意味着频率偏移增大，两个符号的区分度更高，抗噪声能力增强。但是调制指数过大会导致信号带宽增加，可能引起邻道干扰。标准规定范围为 0.45~0.55。sps越大，BER 越低。因为采样点数增加，高斯滤波更精确，波形更平滑，解调性能提升。但是sps 增大意味着计算量线性增加，仿真时间变长。

---

## 课后实践

请补全下方 BER-SNR 扫描循环中的 **3 处空缺**（每处一行代码），完成完整的调制→信道→解调→误码统计链路。

要求：

1. 补全 AWGN 信道加噪
2. 补全 GFSK 解调
3. 补全 BER 计算

完成后运行 `python ber_scan_practice.py`，观察 BER 随 SNR 增加而下降的趋势。

In [ ]:
%%writefile ber_scan_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.phy.gfsk import GFSKModulator, GFSKDemodulator
from nearlink_sdr.phy.channel import ChannelModel

sps = 8
num_bits = 10000
rng = np.random.default_rng(42)
tx_bits = rng.integers(0, 2, num_bits)
mod = GFSKModulator(sps=sps)
demod = GFSKDemodulator(sps=sps)

snr_range = [0, 4, 8, 12]
for snr in snr_range:
    tx_signal = mod.modulate(tx_bits)
    ch = ChannelModel(snr_db=float(snr))
    # ==== 1: 信道加噪（1行）====  （补全）
    rx_signal = 
    # ==== 2: GFSK 解调（1行）====  （补全）
    rx_bits = 
    n = min(len(tx_bits), len(rx_bits))
    # ==== 3: 计算 BER（1行）==== （补全）
    ber = 
    print(f"SNR={snr:2d} dB  BER={ber:.6f}")


执行以下命令进行编译并验证结果：


In [ ]:
!python ber_scan_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/01.03_answer.txt
